***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [6. 成像中的去卷积](6_0_introduction.ipynb)
    * 上一节： [6.4 残图与图像质量](6_4_residuals_and_iqa.ipynb)
    * 下一节： [6.x 延伸阅读与参考文献](6_x_further_reading_and_references.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy import ndimage, optimize

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('6_Deconvolution') if Path('6_Deconvolution').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass


本节以下例子全部使用 notebook 内生成的合成图像，因此不依赖 `astropy` 或外部 FITS 文件。这样做的好处是，我们既能控制噪声与源结构，也能明确比较不同噪声估计方法和检测阈值的表现。


In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass


## 6.5 源搜索

在射电天文里，“源搜索”指的是从数据中测量射电源属性，例如流量密度、位置和形态。本节只讨论图像域中的源搜索，也就是在已经成像并得到复原图之后，怎样从图像里识别候选源并估计其参数。

一套完整的图像域源搜索流程，通常至少包含三步：

1. 估计图像噪声，建立显著性阈值。
2. 找到高于阈值的候选 blob。
3. 对这些 blob 做参数化描述，例如峰值流量、位置、角大小与位置角。

看起来很直接，但真正困难的地方在于：干涉图像中的“噪声”并不总是均匀、高斯、独立的；残余旁瓣、校准误差和未完全去卷积的结构都会污染源搜索。因此，源搜索永远不是一个脱离成像质量而独立存在的问题，它与前一节讨论的残图诊断、局部 RMS 和停止准则是紧密耦合的。


In [ ]:
FWHM_FACTOR = 2.0 * np.sqrt(2.0 * np.log(2.0))


def generalGauss2d(x0, y0, sigmax, sigmay, amp=1.0, theta=0.0):
    rtheta = np.deg2rad(theta)
    a = (np.cos(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.sin(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    b = -(np.sin(2.0 * rtheta)) / (4.0 * sigmax ** 2.0) + (np.sin(2.0 * rtheta)) / (4.0 * sigmay ** 2.0)
    c = (np.sin(rtheta) ** 2.0) / (2.0 * sigmax ** 2.0) + (np.cos(rtheta) ** 2.0) / (2.0 * sigmay ** 2.0)
    return lambda x, y: amp * np.exp(-1.0 * (a * (x - x0) ** 2.0 - 2.0 * b * (x - x0) * (y - y0) + c * (y - y0) ** 2.0))


def gaussian1d(x, amp, mean, sigma):
    return amp * np.exp(-0.5 * ((x - mean) / sigma) ** 2)


def make_noise_image(n=128, sigma=0.05, seed=12):
    rng = np.random.default_rng(seed)
    return sigma * rng.standard_normal((n, n))


def make_catalog_field(n=128):
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    field = np.zeros((n, n), dtype=float)
    catalog = [
        (0.90, 20.0, 24.0, 1.5, 1.8, 10.0),
        (0.75, 32.0, 98.0, 1.8, 1.8, 0.0),
        (0.62, 47.0, 52.0, 2.0, 2.8, 25.0),
        (0.55, 60.0, 20.0, 1.6, 2.0, -15.0),
        (0.48, 76.0, 78.0, 2.4, 2.1, 35.0),
        (0.44, 86.0, 108.0, 1.8, 1.8, 0.0),
        (0.38, 99.0, 61.0, 2.0, 1.6, 40.0),
        (0.34, 111.0, 26.0, 1.7, 1.7, 0.0),
        (0.29, 18.0, 74.0, 1.5, 1.5, 0.0),
        (0.25, 70.0, 95.0, 3.5, 5.0, 30.0),
    ]
    for amp, x0, y0, sx, sy, theta in catalog:
        field += generalGauss2d(x0, y0, sx, sy, amp=amp, theta=theta)(xx, yy)
    return field, catalog


def sigma_std(data):
    return np.std(data)


def sigma_mad_median(data):
    median = np.median(data)
    mad = np.median(np.abs(data - median))
    return 1.4826 * mad


def sigma_mad_mean(data):
    mean = np.mean(data)
    mad = np.mean(np.abs(data - mean))
    return np.sqrt(np.pi / 2.0) * mad


def sigma_negative(data):
    median = np.median(data)
    negative = data[data < median] - median
    mirrored = np.concatenate([negative, -negative])
    return np.std(mirrored)


def fit_gaussian_patch(data, x_start, y_start, psf_pix=2.0):
    ny, nx = data.shape
    yy, xx = np.mgrid[0:ny, 0:nx].astype(float)
    peak_y, peak_x = np.unravel_index(np.argmax(data), data.shape)
    amp0 = np.max(data)
    params0 = np.array([amp0, peak_x, peak_y, psf_pix, psf_pix, 0.0, 0.0], dtype=float)
    lower = np.array([0.0, 0.0, 0.0, 0.7 * psf_pix, 0.7 * psf_pix, -90.0, -0.2])
    upper = np.array([2.5 * amp0 + 1e-6, nx - 1.0, ny - 1.0, 5.0 * psf_pix, 5.0 * psf_pix, 90.0, 0.2])

    def gaussian_model(params):
        amp, x0, y0, sigx, sigy, theta, offset = params
        return generalGauss2d(x0, y0, sigx, sigy, amp=amp, theta=theta)(xx, yy) + offset

    def residuals(params):
        return (gaussian_model(params) - data).ravel()

    result = optimize.least_squares(residuals, params0, bounds=(lower, upper))
    amp, x0_fit, y0_fit, sigx, sigy, theta, offset = result.x
    model = gaussian_model(result.x)
    return {
        'peak': float(amp),
        'x': float(x_start + x0_fit),
        'y': float(y_start + y0_fit),
        'fwhm_x': float(FWHM_FACTOR * sigx),
        'fwhm_y': float(FWHM_FACTOR * sigy),
        'theta': float(theta),
        'offset': float(offset),
        'model': model,
    }


def simple_source_finder(data, peak_sigma=5.0, boundary_sigma=3.0, psf_pix=2.0, pad=4):
    sigma_noise = sigma_negative(data)
    peak_threshold = peak_sigma * sigma_noise
    boundary_threshold = boundary_sigma * sigma_noise

    mask = data > boundary_threshold
    labels, nlabels = ndimage.label(mask)
    work_list = []

    for label in range(1, nlabels + 1):
        ys, xs = np.where(labels == label)
        if len(xs) == 0:
            continue
        if np.max(data[labels == label]) < peak_threshold:
            continue

        x0 = max(int(xs.min()) - pad, 0)
        x1 = min(int(xs.max()) + pad + 1, data.shape[1])
        y0 = max(int(ys.min()) - pad, 0)
        y1 = min(int(ys.max()) + pad + 1, data.shape[0])
        fit = fit_gaussian_patch(data[y0:y1, x0:x1], x0, y0, psf_pix=psf_pix)
        fit['bbox'] = (x0, x1, y0, y1)
        work_list.append(fit)

    work_list.sort(key=lambda row: row['peak'], reverse=True)
    catalog = []
    model = np.zeros_like(data)
    merge_radius = 1.5 * FWHM_FACTOR * psf_pix

    for fit in work_list:
        duplicate = any(np.hypot(fit['x'] - prev['x'], fit['y'] - prev['y']) < merge_radius for prev in catalog)
        if duplicate:
            continue
        x0, x1, y0, y1 = fit['bbox']
        model[y0:y1, x0:x1] += fit['model']
        catalog.append(fit)

    residual = data - model
    return catalog, model, residual, sigma_noise


def match_true_and_found(true_catalog, found_catalog):
    matches = []
    for amp, x, y, _, _, _ in true_catalog:
        best = min(found_catalog, key=lambda row: np.hypot(row['x'] - x, row['y'] - y))
        matches.append({
            'true_peak': amp,
            'found_peak': best['peak'],
            'position_error': float(np.hypot(best['x'] - x, best['y'] - y)),
            'peak_error': float(best['peak'] - amp),
        })
    return matches


def make_variable_noise_demo(n=128, seed=22):
    yy, xx = np.mgrid[0:n, 0:n].astype(float)
    radius = np.sqrt((xx - 64.0) ** 2 + (yy - 64.0) ** 2) / 64.0
    sigma_map = 0.015 + 0.055 * radius ** 2
    rng = np.random.default_rng(seed)
    image = sigma_map * rng.standard_normal((n, n))

    positions = [(40.0, 32.0), (104.0, 100.0)]
    for x0, y0 in positions:
        amp = 6.0 * sigma_map[int(round(y0)), int(round(x0))]
        image += generalGauss2d(x0, y0, 1.8, 1.8, amp=amp, theta=0.0)(xx, yy)
    return image, sigma_map, positions


### 6.5.1 噪声估计

任何源搜索器的第一步都是噪声估计。最简单的办法，是直接对整幅图像求标准差；但只要图里已有源、残余旁瓣或校准伪影，这个估计就会被拉高。

为了先看清“纯噪声”本身，下面从一幅只有高斯随机噪声的图像开始。在这种理想情况下，不管用标准差、`MAD Median`、`MAD Mean` 还是“只用负值像素”的估计，结果都应当大致一致。

这里用到两个常见的稳健换算：

$$
\sigma_{\mathrm{MAD,median}} pprox 1.4826\,\mathrm{median}(|x-	ilde{x}|),
$$

$$
\sigma_{\mathrm{MAD,mean}} pprox \sqrt{\pi/2}\,\mathrm{mean}(|x-ar{x}|).
$$

对于“负值像素”方法，我们先以中值为中心，只取负半边像素，再把它镜像到正半边，从而近似重建没有天体正流量污染时的噪声分布。


In [ ]:
pureNoise = make_noise_image(n=128, sigma=0.05, seed=12)
pureNoiseMethods = {
    'STD': sigma_std(pureNoise),
    'MAD median': sigma_mad_median(pureNoise),
    'MAD mean': sigma_mad_mean(pureNoise),
    'Negative-half': sigma_negative(pureNoise),
}

histogram, bins = np.histogram(pureNoise.ravel(), bins=81)
centres = 0.5 * (bins[:-1] + bins[1:])
peak_count = histogram.max()
mean_value = np.mean(pureNoise)

print('Pure-noise sigma estimates:')
for label, value in pureNoiseMethods.items():
    print(f'  {label:<14s} sigma = {value:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

im0 = axes[0].imshow(pureNoise, origin='lower', cmap='coolwarm', vmin=-0.18, vmax=0.18)
axes[0].set_title('Pure noise image')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

axes[1].plot(centres, histogram, 'ko', markersize=3, label='Pixel histogram')
for label, sigma in pureNoiseMethods.items():
    axes[1].plot(centres, gaussian1d(centres, peak_count, mean_value, sigma), linewidth=2, label=label)
axes[1].set_title('Histogram of pure noise pixels')
axes[1].set_xlabel('Pixel value')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()


这个例子里，四种估计得到的噪声几乎一样，这正是理想高斯噪声应有的表现。问题在于，真实射电图像通常并不是这样的。


In [ ]:
sourceField, trueCatalog = make_catalog_field(n=128)
sourceImage = sourceField + make_noise_image(n=128, sigma=0.05, seed=17)

sourceNoiseMethods = {
    'STD': sigma_std(sourceImage),
    'MAD median': sigma_mad_median(sourceImage),
    'MAD mean': sigma_mad_mean(sourceImage),
    'Negative-half': sigma_negative(sourceImage),
}

histogram, bins = np.histogram(sourceImage.ravel(), bins=121)
centres = 0.5 * (bins[:-1] + bins[1:])
peak_count = histogram.max()
median_value = np.median(sourceImage)

print('Source-contaminated image:')
for label, value in sourceNoiseMethods.items():
    print(f'  {label:<14s} sigma = {value:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

im0 = axes[0].imshow(sourceImage, origin='lower', cmap='viridis', vmin=-0.10, vmax=0.45)
axes[0].set_title('Image with sources plus noise')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

axes[1].plot(centres, histogram, 'ko', markersize=3, label='Pixel histogram')
for label, sigma in sourceNoiseMethods.items():
    axes[1].plot(centres, gaussian1d(centres, peak_count, median_value, sigma), linewidth=2, label=label)
axes[1].set_title('Histogram with source-induced positive tail')
axes[1].set_xlabel('Pixel value')
axes[1].set_ylabel('Count')
axes[1].set_yscale('log')
axes[1].set_ylim(1, None)
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()


一旦图像里加入源，像素直方图就会在正流量一侧拉出明显长尾。此时如果仍然直接使用整幅图像标准差，噪声会被严重高估。在这个合成例子中，`STD` 已经明显偏大，而 `MAD median` 和 `Negative-half` 两种方法更接近真实噪声水平。

这也是实际源搜索器中常见的做法：先用稳健统计量估计背景噪声，再据此定义检测阈值。对于 Stokes I 图像，“负半边噪声”特别有用，因为真实天体辐射主要贡献正流量，而负半边更能代表随机噪声与残余伪影。


### 6.5.2 Blob 检测与特征化

在估计出噪声之后，下一步就是在图像中找出候选源。在一般图像处理里，这叫做 *blob detection*。在综合成像的简单情形下，我们可以把一个 blob 理解为一组相邻像素，它们的空间亮度分布可以用一个二维高斯近似描述。

一个常见的检测策略，是同时定义两个阈值：

* 峰值阈值 $\sigma_{m peak} = n\sigma$：blob 的最大像素值至少要高于多少倍噪声。
* 边界阈值 $\sigma_{m boundary} = m\sigma$：用来界定 blob 在图像里的外延范围，其中 $m<n$。

峰值阈值负责回答“这个 blob 是否足够显著”，边界阈值负责回答“要把哪些像素并入这个 blob”。下面实现一个简化版源搜索器：

1. 用负半边方法估计全局噪声。
2. 把所有高于边界阈值的像素做连通域标记。
3. 对每个连通域，如果其峰值高于峰值阈值，就在其包围盒里拟合一个二维高斯。
4. 把拟合结果写入目录，并构造模型图与残图。

这当然还远不是成熟的生产级源搜索器，但已经足以演示 blob 检测的核心思想。


In [ ]:
catalog, modelImage, residualImage, sigmaNoise = simple_source_finder(
    sourceImage,
    peak_sigma=5.0,
    boundary_sigma=3.0,
    psf_pix=2.0,
    pad=4,
)
matches = match_true_and_found(trueCatalog, catalog)

print(f'Estimated noise sigma                  = {sigmaNoise:.4f}')
print(f'Recovered source count                 = {len(catalog)} / {len(trueCatalog)}')
print(f'Median position error                  = {np.median([m["position_error"] for m in matches]):.3f} pixel')
print(f'Max position error                     = {np.max([m["position_error"] for m in matches]):.3f} pixel')
print(f'Median peak-flux error                 = {np.median([m["peak_error"] for m in matches]):.4f}')
print(f'Residual RMS after source extraction   = {np.sqrt(np.mean(residualImage ** 2)):.4f}')

print('\nRecovered catalog (sorted by peak flux):')
print('  Peak      xpix     ypix    FWHM_x   FWHM_y   theta')
for source in catalog:
    print(f'  {source["peak"]:6.3f}   {source["x"]:7.2f}  {source["y"]:7.2f}   {source["fwhm_x"]:6.2f}   {source["fwhm_y"]:6.2f}   {source["theta"]:6.1f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
vmin, vmax = -0.08, 0.45

im0 = axes[0].imshow(sourceImage, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
for _, x0, y0, _, _, _ in trueCatalog:
    axes[0].plot(x0, y0, marker='x', color='cyan', markersize=7, mew=1.5)
for source in catalog:
    axes[0].add_patch(Ellipse((source['x'], source['y']), width=source['fwhm_x'], height=source['fwhm_y'], angle=source['theta'], fill=False, edgecolor='white', linewidth=1.2))
axes[0].set_title('Data with true and recovered sources')
axes[0].set_xlabel('Pixel')
axes[0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0], shrink=0.78)

im1 = axes[1].imshow(modelImage, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title('Gaussian source model')
axes[1].set_xlabel('Pixel')
axes[1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[1], shrink=0.78)

im2 = axes[2].imshow(residualImage, origin='lower', cmap='coolwarm', vmin=-0.16, vmax=0.16)
axes[2].set_title('Residual after source extraction')
axes[2].set_xlabel('Pixel')
axes[2].set_ylabel('Pixel')
fig.colorbar(im2, ax=axes[2], shrink=0.78)

plt.tight_layout()


*图：左图是输入数据，青色叉号表示真实源位置，白色椭圆表示源搜索器恢复出的高斯参数；中图是拟合得到的模型图；右图是减去模型后的残图。对于这些彼此分离较好的源，这个简化算法已经能给出相当合理的位置与大小估计。*


但这个算法仍然有明显局限：

* 如果多个源彼此很近，就可能被合并成一个 blob，或者拟合结果相互干扰。
* 如果背景噪声并不均匀，全局单一阈值就会失效。
* 如果图像里还残留旁瓣、负碗或方向相关伪影，源搜索器很容易把它们误认成真实源。
* 对于明显展源或非高斯形态源，用单个二维高斯去拟合也不再合适。

这些正是成熟源搜索软件要解决的问题：deblending、局部噪声估计、多尺度建模、参数约束以及更复杂的误差传播。


### 6.5.3 全局阈值与局部阈值

上面的简化源搜索器使用的是一个全局噪声 $\sigma$。这在噪声相对均匀时可以工作，但在真实干涉图像中，噪声往往并不平坦。例如：

* 主波束校正后，边缘区域噪声会显著升高。
* 亮源附近的去卷积残差会抬高局部 RMS。
* 拼接（mosaic）图像和不同权重方案也会带来位置相关的噪声变化。

下面给出一个专门的合成例子。我们人为构造一幅“边缘更吵、中心更安静”的图像，并在不同位置放入两个具有相同局部显著性的点源。由于全局标准差会被高噪声边缘主导，使用单一的全局 `5σ` 阈值时，中心较安静区域的源反而可能被漏掉；而若按局部噪声归一化成 S/N 图，两者又都会重新变得显著。


In [ ]:
variableNoiseImage, trueSigmaMap, variableSources = make_variable_noise_demo(n=128, seed=22)
globalSigma = sigma_negative(variableNoiseImage)
globalSnrMap = variableNoiseImage / globalSigma
localSnrMap = variableNoiseImage / trueSigmaMap

print(f'Global noise estimate = {globalSigma:.4f}')
for idx, (x0, y0) in enumerate(variableSources, start=1):
    peak = variableNoiseImage[int(round(y0)), int(round(x0))]
    print(
        f'  Source {idx}: peak = {peak:.4f} | '
        f'global S/N = {peak / globalSigma:.2f} | '
        f'local S/N = {peak / trueSigmaMap[int(round(y0)), int(round(x0))]:.2f}'
    )

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

im0 = axes[0, 0].imshow(variableNoiseImage, origin='lower', cmap='viridis')
axes[0, 0].set_title('Image with spatially varying noise')
axes[0, 0].set_xlabel('Pixel')
axes[0, 0].set_ylabel('Pixel')
fig.colorbar(im0, ax=axes[0, 0], shrink=0.8)

im1 = axes[0, 1].imshow(trueSigmaMap, origin='lower', cmap='magma')
axes[0, 1].set_title('True local noise map')
axes[0, 1].set_xlabel('Pixel')
axes[0, 1].set_ylabel('Pixel')
fig.colorbar(im1, ax=axes[0, 1], shrink=0.8)

im2 = axes[1, 0].imshow(globalSnrMap, origin='lower', cmap='coolwarm', vmin=-8, vmax=8)
axes[1, 0].contour(globalSnrMap, levels=[5.0], colors='k', linewidths=1.0)
axes[1, 0].set_title('Global S/N map with 5σ contour')
axes[1, 0].set_xlabel('Pixel')
axes[1, 0].set_ylabel('Pixel')
fig.colorbar(im2, ax=axes[1, 0], shrink=0.8)

im3 = axes[1, 1].imshow(localSnrMap, origin='lower', cmap='coolwarm', vmin=-8, vmax=8)
axes[1, 1].contour(localSnrMap, levels=[5.0], colors='k', linewidths=1.0)
axes[1, 1].set_title('Local S/N map with 5σ contour')
axes[1, 1].set_xlabel('Pixel')
axes[1, 1].set_ylabel('Pixel')
fig.colorbar(im3, ax=axes[1, 1], shrink=0.8)

for ax in axes.ravel():
    for idx, (x0, y0) in enumerate(variableSources, start=1):
        ax.plot(x0, y0, marker='o', markersize=6, markerfacecolor='none', markeredgecolor='white', markeredgewidth=1.5)
        ax.text(x0 + 2, y0 + 2, f'S{idx}', color='white', fontsize=10)

plt.tight_layout()


这个例子非常典型：如果只看全局 `5σ` 阈值，靠近安静区域的 `S1` 会被低估显著性，而高噪声区域里的 `S2` 反而因为绝对流量更高而容易过阈值；一旦按局部噪声归一化成 S/N 图，两者又回到相近的局部显著性水平。这就是为什么成熟源搜索器通常都要先估计局部 RMS 图，再在局部 S/N 图而不是原始亮度图上做检测。

当然，在真实数据中我们并不知道这里那样的“真局部噪声图”，必须用滑动窗口、源掩膜和稳健统计量去估计。这个估计本身就与前一节讨论的残图质量直接相关：如果残图中仍保留强相关结构，那么局部 RMS 图也会被污染，从而进一步影响源搜索。


这一节最重要的结论是：源搜索不是“阈值一下就结束”的简单操作。它至少依赖三个前提：

* 噪声估计要尽可能稳健。
* 检测阈值最好基于局部 S/N，而不是单一全局亮度阈值。
* 拟合模型必须与源形态和图像分辨率相匹配。

当这些前提不满足时，源目录中的偏差、漏检和误检都会迅速增加。也正因如此，现代射电数据处理中，源搜索通常被视作“成像之后的第二次建模”，而不仅仅是一道后处理步骤。


***

* 下一节： [6.x 延伸阅读与参考文献](6_x_further_reading_and_references.ipynb)
